# llm-eval-gate · 실제 Gemini로 평가하기

API 키를 저장소에 올리지 않고 실제 모델로 평가하는 노트북입니다.

- 키는 **Colab 보안 비밀**(왼쪽 🔑 아이콘)에만 저장되고, 노트북·저장소·결과 파일 어디에도 남지 않습니다
- 결과로 나오는 `eval_results.zip`에는 모델 응답 캐시와 리포트만 들어 있습니다
- 이 zip을 저장소에 반영하면 GitHub CI가 **키 없이** 같은 응답을 재생해 게이트를 돌립니다

### 처음 한 번만
1. [aistudio.google.com](https://aistudio.google.com)에서 무료 API 키를 발급합니다
2. 이 노트북 왼쪽의 🔑(보안 비밀) → 새 보안 비밀 추가 → 이름 `GEMINI_API_KEY`, 값에 키를 넣고 **노트북 액세스**를 켭니다

## 1. 설정

In [ ]:
# 이번 결과를 기준선(baseline.json)으로 저장할지. 첫 실행은 False로 두고 결과를 먼저 확인하세요.
UPDATE_BASELINE = False

import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("API key loaded from Colab secrets:", bool(os.environ["GEMINI_API_KEY"]))

## 2. 저장소 받기와 설치

In [ ]:
import os

if not os.path.exists("/content/llm-eval-gate"):
    !git clone -q https://github.com/zerda217/llm-eval-gate /content/llm-eval-gate
%cd /content/llm-eval-gate
!git pull -q
!pip install -q -e .

## 3. 평가 실행

무료 한도에 맞춰 분당 10건씩 호출하므로 40건에 4~5분 걸립니다. 이미 캐시에 있는 케이스는 다시 호출하지 않습니다.

마지막 줄의 `exit 1`은 게이트가 막혔다는 뜻입니다. 오류가 아니라 평가 결과입니다.

In [ ]:
args = "--mode cached --json-out eval.json --md-out eval.md"
if UPDATE_BASELINE:
    args += " --update-baseline"
!llm-eval-gate run examples/schedule/eval.yaml {args}; echo "exit $?"

## 4. 결과 파일 만들기

In [ ]:
import glob, os, zipfile
from google.colab import files

paths = glob.glob("examples/schedule/cache/**/*.json", recursive=True)
paths += [p for p in ["eval.json", "eval.md", "examples/schedule/baseline.json"] if os.path.exists(p)]

# 안전 확인: 결과 파일 어디에도 API 키가 없어야 합니다
key = os.environ["GEMINI_API_KEY"]
leaked = [p for p in paths if key and key in open(p, encoding="utf-8").read()]
assert not leaked, f"API key found in: {leaked}"

with zipfile.ZipFile("eval_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in paths:
        zf.write(p)
print(f"{len(paths)} files packed, API key not included")
files.download("eval_results.zip")

## 5. 저장소에 반영

다운로드된 `eval_results.zip`을 Claude와의 대화에 첨부하면, 캐시만 커밋해 저장소에 올립니다.
그 뒤부터 GitHub의 `eval-replay` 단계가 키 없이 이 결과로 게이트를 돌립니다.

프롬프트(`system.txt`, `prompt.txt`)나 모델을 바꾸면 캐시 키가 달라지므로, 이 노트북을 다시 실행해 새 결과를 반영해야 합니다.